# Rocket Fast-DeePC Video

Fixed benchmark: $T=225$, initial condition (0.65, 0.92, 0.05). This notebook times the three controllers once and records the original and C Fast-DeePC landings.


## 1. Import packages and locate the repository


This example uses Python 3.10. Run the following commands in PowerShell:


git clone https://github.com/TerryZhajimi/Fast-DeePC.git
cd Fast-DeePC
git clone https://github.com/michael-cummins/DeePC-Hunt.git external/DeePC-Hunt-main

py -3.10 -m venv .venv
.\.venv\Scripts\Activate.ps1

python -m pip install -r requirements.txt

python -m pip install -e .\external\DeePC-Hunt-main

python -m pip install "gymnasium==0.27.1" "pygame==2.1.3.dev8" "moviepy<2"

python -m pip install --only-binary=:all: "Box2D==2.3.10"

python -m pip install --no-deps "git+https://git.ethz.ch/bsaverio/coco-project.git"


pwsh -ExecutionPolicy Bypass -File .\scripts\build_windows.ps1
jupyter notebook

In [ ]:
from pathlib import Path
import ctypes as ct
import os
import random
import time
import warnings

# Add the optional MSYS2 runtime only when it exists on Windows.
MSYS2_DLL_DIR = Path(r"C:\msys64\ucrt64\bin")
_msys2_dll_directory = None
if os.name == "nt" and hasattr(os, "add_dll_directory") and MSYS2_DLL_DIR.is_dir():
    _msys2_dll_directory = os.add_dll_directory(str(MSYS2_DLL_DIR))

import cvxpy as cp
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import cho_factor, cho_solve

import coco_rocket_lander  # Registers coco_rocket_lander/RocketLander-v0.
from deepc_hunt.controllers import npDeePC

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.set_printoptions(precision=6, suppress=True)

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
PROJECT_DIR = ROOT
ROCKET_REPO = Path(os.environ.get("DEEPC_HUNT_DIR", ROOT / "external" / "DeePC-Hunt-main"))
DATA_DIR = ROCKET_REPO / "examples" / "data"

DLL_CANDIDATES = [
    ROOT / "build" / "deepc_boxqp_optimized.dll",
]
DLL_PATH = next((path for path in DLL_CANDIDATES if path.exists()), None)
if DLL_PATH is None:
    raise FileNotFoundError("deepc_boxqp_optimized.dll was not found")

print("Python environment ready")
print("CVXPY:", cp.__version__)
print("C solver:", DLL_PATH)


## 2. Define the fixed rocket, DeePC, and constraint settings


In [ ]:
# Shared DeePC and rocket settings.
T_INI = 1
HORIZON = 10
DIM_U = 3
DIM_Y = 6
NU = HORIZON * DIM_U
NY = HORIZON * DIM_Y
NZ = NU + NY

# Quadratic Fast-DeePC weights selected by a C-only landing sweep. The larger lateral-input and pitch penalties prevent the aggressive side-force saturation that caused the earlier crash.
Q_STEP = np.diag([100.0, 15.0, 25.0, 2.2, 6000.0, 120.0])
R_STEP = np.diag([0.01, 0.30, 0.30])
WY = 2.0 * np.kron(np.eye(HORIZON), Q_STEP)
WU = 2.0 * np.kron(np.eye(HORIZON), R_STEP)

# Preserve the exact paper weights for the original DeePC-Hunt baseline.
ORIGINAL_Q_STEP = np.diag([100.0, 10.0, 5.0, 1.0, 3000.0, 30.0])
ORIGINAL_R_STEP = np.diag([0.01, 0.01, 0.01])

# Fixed quadratic Fast-DeePC penalties.
LAMBDA_PEN = 1.0e5
RHO_PEN = 1.0e5
LAMBDA_G = 281.1706625951745

# Parameters used by the trained NumPy DeePC-Hunt controller in examples/rocket.ipynb.

ORIGINAL_PARAMS = {
    "lam_g1": 50.0,
    "lam_g2": 10.3,
    "lam_y": 1000.0,
}

U_LOWER_STEP = np.array([0.0, -1.0, -1.0])
U_UPPER_STEP = np.array([1.0, 1.0, 1.0])
PHYSICAL_Y_LOWER_STEP = np.array([0.0, 7.0, -100.0, -100.0, -0.6, -100.0])
PHYSICAL_Y_UPPER_STEP = np.array([33.0, 26.6, 100.0, 100.0, 0.6, 100.0])
# A 0.15-rad prediction margin protects the nonlinear plant's physical pitch box. The original controller continues to use the physical +/-0.6 bound.
Y_LOWER_STEP = np.array([0.0, 7.0, -100.0, -100.0, -0.45, -100.0])
Y_UPPER_STEP = np.array([33.0, 26.6, 100.0, 100.0, 0.45, 100.0])

U_LOWER = np.tile(U_LOWER_STEP, HORIZON)
U_UPPER = np.tile(U_UPPER_STEP, HORIZON)
Y_LOWER = np.tile(Y_LOWER_STEP, HORIZON)
Y_UPPER = np.tile(Y_UPPER_STEP, HORIZON)
ORIGINAL_Y_LOWER = np.tile(PHYSICAL_Y_LOWER_STEP, HORIZON)
ORIGINAL_Y_UPPER = np.tile(PHYSICAL_Y_UPPER_STEP, HORIZON)
Z_LOWER = np.concatenate([U_LOWER, Y_LOWER])
Z_UPPER = np.concatenate([U_UPPER, Y_UPPER])
U_REF = np.zeros(NU)

# Same-problem CVXPY/C accuracy settings.
SOLVER_TOL = 1.0e-5
MAX_SOLVER_ITER = 100000
C_MAX_ITER = 100

# Original problem is unchanged. These are solver-only runtime settings.
ORIGINAL_SOLVER_TOL = 1.0e-4
ORIGINAL_MAX_ITER = 10000
ORIGINAL_POLISHING = False

# One fixed, reproducible landing experiment.
FIXED_INITIAL_POSITION = (0.65, 0.92, 0.05)
FIXED_ENV_SEED = 42
BENCHMARK_SEEDS = [FIXED_ENV_SEED]
all_seeds = BENCHMARK_SEEDS
MAX_STEPS = 600
LOCKSTEP_MAX_STEPS = 80
RUN_FULL_BENCHMARK = False
ENABLE_ORIGINAL_BASELINE = True



## 3. Load the fixed $T=225$ offline trajectory and form the Hankel matrices


In [ ]:
#order will be tini + n which is L
def block_hankel(data, order):
    data = np.asarray(data, dtype=np.float64)
    samples, channels = data.shape
    ncol = samples - order + 1
    if ncol <= 0:
        raise ValueError(f"Need T >= {order}; received T={samples}")
    H = np.empty((order * channels, ncol), dtype=np.float64)
    for column in range(ncol):
        H[:, column] = data[column:column + order].reshape(-1)
    return H


u_data = np.loadtxt(DATA_DIR / "rocket_ud.csv", delimiter=",", dtype=np.float64)
y_data = np.loadtxt(DATA_DIR / "rocket_yd.csv", delimiter=",", dtype=np.float64)
assert u_data.shape[1] == DIM_U
assert y_data.shape[1] == DIM_Y
assert len(u_data) == len(y_data)
assert len(u_data) == 225, "This notebook intentionally keeps T=225"

L = T_INI + HORIZON
HU = block_hankel(u_data, L)
HY = block_hankel(y_data, L)
UP = HU[:T_INI * DIM_U]
UF = HU[T_INI * DIM_U:]
YP = HY[:T_INI * DIM_Y]
YF = HY[T_INI * DIM_Y:]
Z = np.vstack([UP, YP, UF])
NCOL = UF.shape[1]

assert UF.shape == (NU, NCOL)
assert YF.shape == (NY, NCOL)
print("Data length T:", len(u_data))
print("g dimension:", NCOL)
print("Reduced online dimension N(m+p):", NZ)
print("Hankel block shapes:", UP.shape, YP.shape, UF.shape, YF.shape)

def initial_position_from_seed(seed):
    # The seed controls simulator randomness; position is fixed.
    return FIXED_INITIAL_POSITION


def make_environment(seed, initial_position=None):
    if initial_position is None:
        initial_position = initial_position_from_seed(seed)
    env = gym.make(
        "coco_rocket_lander/RocketLander-v0",
        args={"initial_position": initial_position},
    )
    obs, info = env.reset(seed=seed)
    return env, np.asarray(obs, dtype=np.float64), info


def reference_from_landing(landing_position):
    step_ref = np.array([
        landing_position[0],
        landing_position[1],
        0.0,
        0.0,
        landing_position[2],
        0.0,
    ], dtype=np.float64)
    return np.tile(step_ref, HORIZON)


def update_window(u_window, y_window, action, output_before_action):
    u_next = np.vstack([u_window[1:], np.asarray(action).reshape(1, DIM_U)])
    y_next = np.vstack([y_window[1:],np.asarray(output_before_action).reshape(1, DIM_Y)])
    return u_next, y_next



## 4. Build the full soft-L2 CVXPY baseline


In [ ]:
class FullQuadraticCVXPY:
    name = "Full quadratic CVXPY"

    def __init__(self):
        start = time.perf_counter()
        self.g = cp.Variable(NCOL)
        self.u = cp.Variable(NU)
        self.y = cp.Variable(NY)
        self.u_ini = cp.Parameter(T_INI * DIM_U)
        self.y_ini = cp.Parameter(T_INI * DIM_Y)
        self.y_ref = cp.Parameter(NY)

        target = cp.hstack([self.u_ini, self.y_ini, self.u])
        objective = 0.5 * cp.sum(cp.multiply(np.diag(WU), cp.square(self.u - U_REF)))
        objective += 0.5 * cp.sum(cp.multiply(np.diag(WY), cp.square(self.y - self.y_ref)))
        objective += 0.5 * LAMBDA_PEN * cp.sum_squares(Z @ self.g - target)
        objective += 0.5 * RHO_PEN * cp.sum_squares(YF @ self.g - self.y)
        objective += 0.5 * LAMBDA_G * cp.sum_squares(self.g)

        constraints = [
            self.u >= U_LOWER,
            self.u <= U_UPPER,
            self.y >= Y_LOWER,
            self.y <= Y_UPPER,
        ]
        self.problem = cp.Problem(cp.Minimize(objective), constraints)
        if not self.problem.is_qp():
            raise RuntimeError("The full quadratic baseline was not recognized as a QP")
        self.offline_time = time.perf_counter() - start

    def reset(self):
        for variable in (self.g, self.u, self.y):
            variable.value = None

    def solve(self, u_ini, y_ini, y_ref):
        start = time.perf_counter()
        self.u_ini.value = np.asarray(u_ini).reshape(-1)
        self.y_ini.value = np.asarray(y_ini).reshape(-1)
        self.y_ref.value = np.asarray(y_ref).reshape(-1)
        self.problem.solve(
            solver=cp.OSQP,
            warm_start=True,
            eps_abs=SOLVER_TOL,
            eps_rel=SOLVER_TOL,
            max_iter=MAX_SOLVER_ITER,
            polishing=False,
            adaptive_rho=True,
            check_termination=25,
            verbose=False,
        )
        online_time = time.perf_counter() - start
        if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(f"Full CVXPY status: {self.problem.status}")
        return {
            "u": np.asarray(self.u.value, dtype=np.float64).reshape(-1).copy(),
            "y": np.asarray(self.y.value, dtype=np.float64).reshape(-1).copy(),
            "g": np.asarray(self.g.value, dtype=np.float64).reshape(-1).copy(),
            "objective": float(self.problem.value),
            "online_time": online_time,
            "solver_time": float(self.problem.solver_stats.solve_time or 0.0),
            "status": self.problem.status,
            "iterations": int(self.problem.solver_stats.num_iters or 0),
        }



## 5. Build the C Fast-DeePC BoxQP controller


In [ ]:
BINT = ct.c_long


class BoxQPInfo(ct.Structure):
    _fields_ = [
        ("iterations", BINT),
        ("mu", ct.c_double),
        ("duality", ct.c_double),
        ("status", ct.c_int),
        ("chol_info", BINT),
    ]


lib = ct.CDLL(str(DLL_PATH))
double_p = ct.POINTER(ct.c_double)
lib.dense_boxqp_solve.argtypes = [
    double_p, double_p, BINT, ct.c_double, BINT,
    double_p, double_p, ct.POINTER(BoxQPInfo),
]
"""
#dense_boxqp_solve(
    const double *H,
    const double *h,
    BINT n,
    double epsilon,
    BINT max_iter,
    const double *z_warm,
    double *z_out,
    BoxQPInfo *info
)
"""
lib.dense_boxqp_solve.restype = ct.c_int


def ptr(array):
    return array.ctypes.data_as(double_p)


class ReducedCBoxQP:
    name = "Reduced C BoxQP"

    def __init__(self):
        start = time.perf_counter()
        self.M = (
            LAMBDA_PEN * (Z.T @ Z)
            + RHO_PEN * (YF.T @ YF)
            + LAMBDA_G * np.eye(NCOL)
        )
        self.M_factor = cho_factor(self.M, lower=True, check_finite=False)
        #precompute with further offline elimination, the first one is p
        z_p = np.vstack([UP, YP])
        self.ini_to_q = cho_solve(
            self.M_factor, LAMBDA_PEN * z_p.T, check_finite=False
        )
        #this is su and sy
        self.ini_to_h = np.vstack([
            -LAMBDA_PEN * UF @ self.ini_to_q,
            -RHO_PEN * YF @ self.ini_to_q,
        ])
        #the hessain matrix build 
        coupling = np.hstack([LAMBDA_PEN * UF.T, RHO_PEN * YF.T])

        H0 = np.zeros((NZ, NZ), dtype=np.float64)
        H0[:NU, :NU] = WU + LAMBDA_PEN * np.eye(NU)
        H0[NU:, NU:] = WY + RHO_PEN * np.eye(NY)
        #m-1B
        solved_coupling = cho_solve(self.M_factor, coupling, check_finite=False)
        self.H_hat = H0 - coupling.T @ solved_coupling
        self.H_hat = 0.5 * (self.H_hat + self.H_hat.T)
        #convert bound to -1 and 1
        self.box_center = 0.5 * (Z_UPPER + Z_LOWER)
        self.box_scale = 0.5 * (Z_UPPER - Z_LOWER)
        self.h_center = self.H_hat @ self.box_center
        self.H_box = self.box_scale[:, None] * self.H_hat * self.box_scale[None, :]
        self.H_box = np.asfortranarray(0.5 * (self.H_box + self.H_box.T))
        if np.linalg.eigvalsh(self.H_box)[0] <= 0:
            raise RuntimeError("Reduced Hessian is not positive definite")
        self.s_warm = None
        self.offline_time = time.perf_counter() - start

    def reset(self):
        self.s_warm = None

    def solve(self, u_ini, y_ini, y_ref):
        u_ini = np.asarray(u_ini, dtype=np.float64).reshape(-1)
        y_ini = np.asarray(y_ini, dtype=np.float64).reshape(-1)
        y_ref = np.asarray(y_ref, dtype=np.float64).reshape(-1)

        start = time.perf_counter()
        #eq 26 find the ht 
        ini = np.concatenate([u_ini, y_ini])
        h_hat = np.concatenate([-WU @ U_REF, -WY @ y_ref])
        h_hat += self.ini_to_h @ ini
        h_box = self.box_scale * (self.h_center + h_hat)
        h_box = np.ascontiguousarray(h_box, dtype=np.float64)

        s_out = np.zeros(NZ, dtype=np.float64)
        info = BoxQPInfo()
        warm = None
        if self.s_warm is not None:
            self.s_warm = np.ascontiguousarray(self.s_warm, dtype=np.float64)
            warm = ptr(self.s_warm)
        status = lib.dense_boxqp_solve(
            ptr(self.H_box),
            ptr(h_box),
            BINT(NZ),
            ct.c_double(SOLVER_TOL),
            BINT(C_MAX_ITER),
            warm,
            ptr(s_out),
            ct.byref(info),
        )
        online_time = time.perf_counter() - start
        if status not in (0, 1):
            raise RuntimeError(f"C status={status}, chol_info={info.chol_info}")

        z = self.box_center + self.box_scale * s_out
        u = z[:NU]
        y = z[NU:]
        self.s_warm = s_out.copy()
        return {
            "u": u.copy(),
            "y": y.copy(),
            "online_time": online_time,
            "solver_time": online_time,
            "status": int(status),
            "iterations": int(info.iterations),
            "mu": float(info.mu),
        }



## 6. Build the original hard-L1 DeePC-Hunt baseline and warm up all three controllers


In [ ]:
class OriginalDeePCHunt:
    name = "Original DeePC-Hunt"

    def __init__(self):
        start = time.perf_counter()
        self.controller = npDeePC(
            ud=u_data,
            yd=y_data,
            u_constraints=(U_LOWER, U_UPPER),
            y_constraints=(ORIGINAL_Y_LOWER, ORIGINAL_Y_UPPER),
            Tini=T_INI,
            N=HORIZON,
            m=DIM_U,
            p=DIM_Y,
            n=DIM_Y,
        )
        self.controller.setup(
            Q=ORIGINAL_Q_STEP,
            R=ORIGINAL_R_STEP,
            lam_g1=ORIGINAL_PARAMS["lam_g1"],
            lam_g2=ORIGINAL_PARAMS["lam_g2"],
            lam_y=ORIGINAL_PARAMS["lam_y"],
        )
        if not self.controller.problem.is_qp():
            raise RuntimeError("Original DeePC-Hunt problem is not a QP")
        self.offline_time = time.perf_counter() - start

    def reset(self):
        for variable in self.controller.problem.variables():
            variable.value = None

    def solve(self, u_ini, y_ini, y_ref):
        start = time.perf_counter()
        self.controller.u_ini.value = np.asarray(u_ini).reshape(-1)
        self.controller.y_ini.value = np.asarray(y_ini).reshape(-1)
        self.controller.y_ref.value = np.asarray(y_ref).reshape(-1)
        self.controller.u_ref.value = U_REF
        self.controller.problem.solve(
            solver=cp.OSQP,
            warm_start=True,
            eps_abs=ORIGINAL_SOLVER_TOL,
            eps_rel=ORIGINAL_SOLVER_TOL,
            max_iter=ORIGINAL_MAX_ITER,
            polishing=ORIGINAL_POLISHING,
            adaptive_rho=True,
            check_termination=25,
            verbose=False,
        )
        online_time = time.perf_counter() - start
        if self.controller.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(f"Original DeePC-Hunt status: {self.controller.problem.status}")
        return {
            "u": np.asarray(self.controller.u.value, dtype=np.float64).reshape(-1).copy(),
            "y": np.asarray(self.controller.y.value, dtype=np.float64).reshape(-1).copy(),
            "g": np.asarray(self.controller.g.value, dtype=np.float64).reshape(-1).copy(),
            "objective": float(self.controller.problem.value),
            "online_time": online_time,
            "solver_time": float(self.controller.problem.solver_stats.solve_time or 0.0),
            "status": self.controller.problem.status,
            "iterations": int(self.controller.problem.solver_stats.num_iters or 0),
        }

cvx_controller = FullQuadraticCVXPY()
c_controller = ReducedCBoxQP()
original_controller = OriginalDeePCHunt() if ENABLE_ORIGINAL_BASELINE else None

warm_seed = all_seeds[0]
warm_env, warm_obs, _ = make_environment(warm_seed)
warm_landing = np.asarray(warm_env.get_landing_position(), dtype=np.float64)
warm_ref = reference_from_landing(warm_landing)
warm_u_ini = np.zeros(T_INI * DIM_U)
warm_y_ini = np.tile(warm_obs[:DIM_Y], T_INI)
warm_env.close()

warmup_times = {}
for controller in [cvx_controller, c_controller, original_controller]:
    if controller is None:
        continue
    start = time.perf_counter()
    result = controller.solve(warm_u_ini, warm_y_ini, warm_ref)
    warmup_times[controller.name] = time.perf_counter() - start
    controller.reset()

print("Offline construction times (s):")
for controller in [cvx_controller, c_controller, original_controller]:
    if controller is not None:
        print(f"  {controller.name}: {controller.offline_time:.6f}")
print("Untimed warm-up solve times (s):", warmup_times)


## 7. Solve the one fixed landing case


In [ ]:
# Environment and closed-loop simulation helpers.

VIDEO_DIR = ROOT / "media" / "generated_rocket_videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_SEED = FIXED_ENV_SEED  # Used later when replaying the videos.


def make_environment_at(
    seed,
    initial_position,
    render_mode=None,
    record_video=False,
    video_subdir=None,
    name_prefix="rocket",
):
    env = gym.make(
        "coco_rocket_lander/RocketLander-v0",
        render_mode=render_mode,
        args={
            "initial_position": tuple(initial_position),
            "enable_wind": False,
        },
    )

    if record_video:
        video_subdir.mkdir(parents=True, exist_ok=True)
        env = gym.wrappers.RecordVideo(
            env,
            video_folder=str(video_subdir),
            episode_trigger=lambda _: True,
            name_prefix=name_prefix,
        )

    obs, info = env.reset(seed=seed)
    return env, np.asarray(obs, dtype=float), info


def run_episode_at_initial(controller, seed, initial_position, max_steps):
    env, obs, _ = make_environment_at(seed, initial_position)
    landing = np.asarray(env.get_landing_position(), dtype=float)
    y_ref = reference_from_landing(landing)
    controller.reset()

    u_window = np.zeros((T_INI, DIM_U))
    y_window = np.tile(obs[:DIM_Y], (T_INI, 1))

    states = [obs[:DIM_Y].copy()]
    actions, solve_times, iterations = [], [], []
    failure_message = None

    for _ in range(max_steps):
        output_before_action = obs[:DIM_Y].copy()

        try:
            result = controller.solve(
                u_window.ravel(),
                y_window.ravel(),
                y_ref,
            )
            action = result["u"][:DIM_U].copy()
            solve_times.append(result.get("online_time", np.nan))
            iterations.append(result.get("iterations", np.nan))
        except Exception as error:
            failure_message = repr(error)
            break

        obs, _, terminated, truncated, _ = env.step(action)
        obs = np.asarray(obs, dtype=float)

        actions.append(action)
        states.append(obs[:DIM_Y].copy())
        u_window, y_window = update_window(
            u_window,
            y_window,
            action,
            output_before_action,
        )

        if terminated or truncated:
            break

    env.close()

    success = bool(obs[6]) and bool(obs[7]) and obs[1] >= landing[1]

    return {
        "controller": controller.name,
        "initial_position": tuple(initial_position),
        "landing": landing,
        "states": np.asarray(states),
        "actions": np.asarray(actions),
        "solve_times": np.asarray(solve_times),
        "iterations": np.asarray(iterations),
        "success": success,
        "episode_length": len(actions),
        "failure_message": failure_message,
    }

In [ ]:
selected_video_case = {
    "initial": FIXED_INITIAL_POSITION,
    "original": run_episode_at_initial(original_controller, FIXED_ENV_SEED, FIXED_INITIAL_POSITION, MAX_STEPS),
    "c": run_episode_at_initial(c_controller, FIXED_ENV_SEED, FIXED_INITIAL_POSITION, MAX_STEPS),
}
orig_plot_ep = selected_video_case["original"]
c_plot_ep = selected_video_case["c"]
print("Fixed initial condition:", FIXED_INITIAL_POSITION)
print("Original success / steps:", orig_plot_ep["success"], orig_plot_ep["episode_length"])
print("C Fast-DeePC success / steps:", c_plot_ep["success"], c_plot_ep["episode_length"])


## 8. Replay the solved actions and save the two landing videos


In [ ]:
# Record videos by replaying the exact actions used in the preceding trajectory plot.

def replay_actions_to_video(solved_episode, video_name):
    video_subdir = VIDEO_DIR / video_name
    env, observation, _ = make_environment_at(
        seed=SEARCH_SEED,
        initial_position=selected_video_case["initial"],
        render_mode="rgb_array",
        record_video=True,
        video_subdir=video_subdir,
        name_prefix=video_name,
    )

    replayed_states = [observation[:DIM_Y].copy()]
    for action in solved_episode["actions"]:
        observation, _, terminated, truncated, _ = env.step(action)
        observation = np.asarray(observation, dtype=np.float64)
        replayed_states.append(observation[:DIM_Y].copy())
        if terminated or truncated:
            break

    env.close()
    replayed_states = np.asarray(replayed_states, dtype=np.float64)
    solved_states = np.asarray(solved_episode["states"], dtype=np.float64)

    if replayed_states.shape != solved_states.shape:
        raise RuntimeError(
            f"Video replay shape {replayed_states.shape} does not match solved trajectory {solved_states.shape}."
        )

    maximum_state_error = float(np.max(np.abs(replayed_states - solved_states)))
    if maximum_state_error > 1.0e-7:
        raise RuntimeError(
            f"Video replay differs from plotted trajectory: maximum state error={maximum_state_error:.3e}."
        )

    mp4_files = sorted(video_subdir.glob("*.mp4"), key=lambda item: item.stat().st_mtime)
    if not mp4_files:
        raise FileNotFoundError(f"No video was created in {video_subdir}")

    recorded_episode = dict(solved_episode)
    recorded_episode["states"] = replayed_states
    recorded_episode["video_dir"] = str(video_subdir)
    return recorded_episode, mp4_files[-1], maximum_state_error


print("Recording Original DeePC-Hunt video from its plotted action sequence...")
orig_video_ep, original_video_path, original_replay_error = replay_actions_to_video(
    orig_plot_ep,
    "original_deepc_hunt",
)

print("Recording C Fast-DeePC video from its plotted action sequence...")
c_video_ep, c_video_path, c_replay_error = replay_actions_to_video(
    c_plot_ep,
    "c_fast_deepc_boxqp",
)

print("Original video:", original_video_path)
print("C Fast-DeePC video:", c_video_path)
print("Maximum video/plot state errors:", original_replay_error, c_replay_error)



# Display the generated videos in the notebook.
from IPython.display import Video, display
from pathlib import Path

for label, episode in [("Original DeePC-Hunt", orig_video_ep), ("C Fast-DeePC BoxQP", c_video_ep)]:
    folder = Path(episode["video_dir"])
    mp4_files = sorted(folder.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    print("\n", label)
    if not mp4_files:
        print("No mp4 file found in", folder)
        continue
    video_path = mp4_files[-1]
    print(video_path)
    display(Video(str(video_path), embed=True, width=520))
